
# Appendix 04: Notation

Source span: printed pages 391-394; PDF pages 404-406. The source was inspected to identify notation categories and symbol families. This notebook does not copy the glossary table. It rebuilds the notation as an executable atlas with original explanations, generated diagrams, and consistency checks.

## Chapter Goal

Notation in directional statistics is not just shorthand. A symbol carries a sample space, an invariance rule, a statistic, and usually a chapter where it first becomes operational. The same letter can be dangerous if the reader forgets whether it refers to a population direction, a sample direction, a concentration parameter, a matrix group, a shape space, or a special function.

This appendix turns notation into a navigable dependency map. Sample spaces are grouped by geometry: circles, spheres, projective spaces, rotation groups, Stiefel frames, Grassmann subspaces, hyperboloids, complex projective spaces, and shape spaces. Population and sample quantities are attached to the spaces on which they make sense. Distribution families are attached to their supports and parameters. Special functions are attached to the estimation or normalization jobs they serve.

The notebook is useful if the reader can answer three questions for each symbol: what object does it denote, what transformation should leave it unchanged or transform it equivariantly, and which chapter uses it as a calculation rather than just a label.



## Visualization Storyboard And Library Routing

- **Notation dependency map.** A NetworkX graph connects sample spaces, statistics, distribution families, special functions, and chapters. This is a proof-of-use view: a symbol should not float without a role.

- **Sample-space gallery.** Matplotlib sketches the main spaces in the glossary: unit circle, unit sphere, projective antipodal identification, a rotation frame, a Stiefel two-frame, and a Grassmann projection. The inspection target is the constraint that defines each space.

- **Distribution-family routing table.** A generated table maps model names to supports, parameters, and diagnostics. It replaces a passive list with a choice guide.

- **Interactive notation graph.** Plotly provides a hoverable graph so the reader can inspect categories and course links without scrolling through a long printed glossary.

The checks are deliberately structural: every symbol entry must have a category, interpretation, support or object type, invariant, and at least one course use. This makes the appendix an auditable course index, not a copied notation page.


In [ ]:

from pathlib import Path
import sys


def find_book_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (
            (candidate / "AGENTS.md").exists()
            and (candidate / "scripts" / "validate_dirstats_course.py").exists()
            and (candidate / "utils").exists()
        ):
            return candidate
    raise RuntimeError("Could not locate Directional-Statistics course root")


BOOK_ROOT = find_book_root(Path.cwd())
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

TOPIC = "appendix-04"
source_span = {"printed": "391-394", "pdf": "404-406"}
print(f"Course root: {BOOK_ROOT}")
source_span


In [ ]:

import csv
import json
import math
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import plotly.graph_objects as go

from utils.artifacts import artifact_path, display_artifact, save_json, save_matplotlib, save_plotly_html
from utils.validation import assert_artifacts

plt.rcParams.update({
    "figure.dpi": 130,
    "axes.grid": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

sample_spaces = [
    {"symbol": "S^1", "name": "unit circle", "constraint": "unit complex number or unit vector in R^2", "chapters": "1-8", "invariant": "angle origin changes rotate all points together"},
    {"symbol": "S^(p-1)", "name": "unit sphere", "constraint": "x has Euclidean norm 1 in R^p", "chapters": "9-12", "invariant": "orthogonal rotations preserve distances and norms"},
    {"symbol": "RP^(p-1)", "name": "real projective space", "constraint": "identify x and -x on the sphere", "chapters": "9-10", "invariant": "sign flips leave an axis unchanged"},
    {"symbol": "O(p), SO(p)", "name": "orthogonal and rotation groups", "constraint": "Q.T Q = I, with determinant +1 for SO(p)", "chapters": "13", "invariant": "basis changes preserve inner products"},
    {"symbol": "V_r(R^p)", "name": "Stiefel manifold", "constraint": "p by r matrices with orthonormal columns", "chapters": "13", "invariant": "frame columns stay orthonormal"},
    {"symbol": "G_r(R^p)", "name": "Grassmann manifold", "constraint": "r-dimensional subspaces represented by projection matrices", "chapters": "13", "invariant": "basis inside the subspace does not matter"},
    {"symbol": "H^(p-1)", "name": "unit hyperboloid", "constraint": "Minkowski norm constraint", "chapters": "13", "invariant": "Lorentz transformations preserve the constraint"},
    {"symbol": "CP^(k-1)", "name": "complex projective space", "constraint": "complex lines through the origin", "chapters": "14", "invariant": "complex phase and scale are nuisance transformations"},
    {"symbol": "Sigma_m^k", "name": "Kendall shape space", "constraint": "centered and scaled landmark configurations modulo rotation", "chapters": "14", "invariant": "translation, scale, and rotation do not change shape"},
]

notation_entries = [
    {"symbol": "mu", "category": "population", "meaning": "mean direction", "object": "S^1 or S^(p-1)", "invariant": "rotates equivariantly with the data", "chapters": "2, 9"},
    {"symbol": "kappa", "category": "population", "meaning": "concentration parameter", "object": "von Mises, Fisher, Watson families", "invariant": "scalar strength, unchanged by rotating coordinates", "chapters": "3, 5, 9, 10"},
    {"symbol": "rho", "category": "population", "meaning": "mean resultant length", "object": "circle or sphere distribution", "invariant": "depends on vector resultant norm", "chapters": "2, 5, 9"},
    {"symbol": "muhat", "category": "sample", "meaning": "sample mean direction", "object": "directional sample", "invariant": "rotates with the sample", "chapters": "2, 5, 10"},
    {"symbol": "R", "category": "sample", "meaning": "resultant length", "object": "sum of unit vectors", "invariant": "norm ignores coordinate basis", "chapters": "2, 4, 6, 10"},
    {"symbol": "Rbar", "category": "sample", "meaning": "mean resultant length", "object": "R divided by sample size", "invariant": "bounded between 0 and 1", "chapters": "2, 5, 10"},
    {"symbol": "T", "category": "sample", "meaning": "scatter or orientation matrix", "object": "spherical observations", "invariant": "transforms by conjugation under rotations", "chapters": "9, 10"},
    {"symbol": "phi_p", "category": "population", "meaning": "trigonometric or characteristic component", "object": "circular distribution", "invariant": "Fourier components track periodic structure", "chapters": "3, 4"},
    {"symbol": "I_p", "category": "special-function", "meaning": "modified Bessel function", "object": "normalizers and Bessel ratios", "invariant": "supports circular and spherical normalizing constants", "chapters": "3, 5, Appendix 01"},
    {"symbol": "J_p", "category": "special-function", "meaning": "Bessel function", "object": "Fourier and distribution theory", "invariant": "oscillatory radial component", "chapters": "4, Appendix 01"},
    {"symbol": "A_p", "category": "special-function", "meaning": "Bessel-ratio resultant calculator", "object": "I_{p/2}/I_{p/2-1}", "invariant": "monotone concentration-to-resultant bridge", "chapters": "5, 10, Appendix 01"},
    {"symbol": "D_p", "category": "special-function", "meaning": "Kummer derivative-ratio calculator", "object": "axial moment equation", "invariant": "maps axial concentration to squared projection moment", "chapters": "10, Appendix 01"},
    {"symbol": "d_F", "category": "shape", "meaning": "full Procrustes distance", "object": "landmark shapes", "invariant": "ignores similarity transformations", "chapters": "14"},
    {"symbol": "H", "category": "shape", "meaning": "Helmert submatrix", "object": "landmark centering transform", "invariant": "removes translation", "chapters": "14"},
    {"symbol": "Z", "category": "shape", "meaning": "preshape or Helmertized coordinates", "object": "centered landmark matrix", "invariant": "unit norm after scaling", "chapters": "14"},
]

distribution_families = [
    {"family": "von Mises", "support": "S^1", "parameters": "mu, kappa", "diagnostic": "A_1 inversion and circular residuals", "chapters": "3, 5, 7"},
    {"family": "cardioid", "support": "S^1", "parameters": "location, first harmonic strength", "diagnostic": "Fourier moment fingerprint", "chapters": "3"},
    {"family": "wrapped normal", "support": "S^1", "parameters": "linear mean and variance before wrapping", "diagnostic": "periodic density reconstruction", "chapters": "3"},
    {"family": "wrapped Cauchy", "support": "S^1", "parameters": "location and concentration-like radius", "diagnostic": "heavy circular tails", "chapters": "3, 5"},
    {"family": "von Mises-Fisher", "support": "S^(p-1)", "parameters": "mean direction and kappa", "diagnostic": "A_p inversion and cap quantiles", "chapters": "9, 10"},
    {"family": "Watson", "support": "RP^(p-1) or S^(p-1)", "parameters": "axis and axial concentration", "diagnostic": "D_p axial moment", "chapters": "9, 10"},
    {"family": "Bingham", "support": "projective or spherical axes", "parameters": "orientation matrix and eigen concentrations", "diagnostic": "scatter eigenstructure", "chapters": "9, 10"},
    {"family": "angular central Gaussian", "support": "S^(p-1)", "parameters": "shape matrix", "diagnostic": "scale-free scatter", "chapters": "9"},
    {"family": "complex Watson/Bingham", "support": "complex projective or shape spaces", "parameters": "complex direction and concentration", "diagnostic": "phase-invariant shape summaries", "chapters": "14"},
]



## Dependency Map: Symbols Should Have Jobs

A printed notation list is alphabetical or categorical. A learning map should be functional. The graph below connects each notation family to the chapters and computations that use it. If a symbol cannot be connected to a sample space, statistic, distribution family, or special function, it is not yet useful notation for the course.

The inspection target is not graph aesthetics. Follow one path. For example, `S^1` connects to circular distribution families, which connect to `A_p`, which connects to concentration estimation. `Sigma_m^k` connects to shape coordinates, Helmert centering, and Procrustes distance. These paths remind the reader that notation is a compressed workflow.


In [ ]:

G = nx.Graph()
for category in ["sample spaces", "population quantities", "sample quantities", "distribution families", "special functions", "shape analysis"]:
    G.add_node(category, kind="category")

for item in sample_spaces:
    node = item["symbol"]
    G.add_node(node, kind="space", label=f"{node}: {item['name']}")
    G.add_edge("sample spaces", node)
    G.add_node(f"chapters {item['chapters']}", kind="chapter")
    G.add_edge(node, f"chapters {item['chapters']}")

for item in notation_entries:
    node = item["symbol"]
    category_node = {
        "population": "population quantities",
        "sample": "sample quantities",
        "special-function": "special functions",
        "shape": "shape analysis",
    }[item["category"]]
    G.add_node(node, kind=item["category"], label=f"{node}: {item['meaning']}")
    G.add_edge(category_node, node)
    G.add_node(f"use {item['chapters']}", kind="chapter")
    G.add_edge(node, f"use {item['chapters']}")

for family in distribution_families:
    node = family["family"]
    G.add_node(node, kind="family", label=f"{node}: {family['support']}")
    G.add_edge("distribution families", node)
    G.add_edge(node, family["support"] if family["support"] in G else "sample spaces")

pos = nx.spring_layout(G, seed=404, k=0.75, iterations=120)
kind_colors = {
    "category": "#263238",
    "space": "#2A9D8F",
    "population": "#E76F51",
    "sample": "#F4A261",
    "special-function": "#6D597A",
    "shape": "#577590",
    "family": "#8AB17D",
    "chapter": "#ADB5BD",
}
node_colors = [kind_colors.get(G.nodes[n].get("kind"), "#CCCCCC") for n in G.nodes]
node_sizes = [950 if G.nodes[n].get("kind") == "category" else 420 for n in G.nodes]

fig, ax = plt.subplots(figsize=(14, 10))
nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.28, width=1.0)
nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors, node_size=node_sizes, linewidths=0.8, edgecolors="white")
labels = {n: n for n in G.nodes if G.nodes[n].get("kind") in {"category", "space", "special-function", "shape", "family"}}
nx.draw_networkx_labels(G, pos, labels=labels, ax=ax, font_size=8)
ax.set_title("Appendix 04 notation dependency map: symbols, spaces, model families, and course uses")
ax.set_axis_off()
fig.tight_layout()
map_path = save_matplotlib(fig, TOPIC, "notation-map", "notation-dependency-map.png")
plt.close(fig)
display_artifact(map_path, width=940)

graph_diagnostics = {
    "graph_node_count": G.number_of_nodes(),
    "graph_edge_count": G.number_of_edges(),
    "all_notation_entries_have_course_use": bool(all(item["chapters"] for item in notation_entries)),
    "all_sample_spaces_have_constraints": bool(all(item["constraint"] for item in sample_spaces)),
}
graph_diagnostics



## Sample-Space Gallery

The glossary names several spaces because directional statistics changes with the space. A circle is periodic; a sphere has rotations and caps; projective data identify antipodal points; rotation groups preserve orthonormal frames; Stiefel points are partial frames; Grassmann points are subspaces; shape spaces remove nuisance similarity transformations.

The gallery below sketches the most common constraints. Each panel is a reminder that an array representation is secondary. The geometry comes first, then the coordinate choice.


In [ ]:

fig = plt.figure(figsize=(13, 8.5))

# S^1
ax = fig.add_subplot(2, 3, 1)
t = np.linspace(0, 2*np.pi, 300)
ax.plot(np.cos(t), np.sin(t), color="#345995")
angles = np.array([0.25, 1.3, 2.8, 4.6])
ax.scatter(np.cos(angles), np.sin(angles), color="#E07A5F", zorder=3)
ax.set_aspect("equal")
ax.set_title("S^1: wrap angles")
ax.set_xticks([]); ax.set_yticks([])

# S^2
ax = fig.add_subplot(2, 3, 2, projection="3d")
nu = np.linspace(0, 2*np.pi, 45)
v = np.linspace(0, np.pi, 24)
U, V = np.meshgrid(nu, v)
X = np.cos(U)*np.sin(V); Y = np.sin(U)*np.sin(V); Z = np.cos(V)
ax.plot_wireframe(X, Y, Z, color="#8ecae6", linewidth=0.35, alpha=0.8)
ax.quiver(0, 0, 0, 0.55, 0.25, 0.78, color="#E76F51", linewidth=2)
ax.set_title("S^2: unit vectors")
ax.set_axis_off(); ax.set_box_aspect((1,1,1))

# Projective antipodal identification
ax = fig.add_subplot(2, 3, 3)
ax.plot(np.cos(t), np.sin(t), color="#adb5bd")
v0 = np.array([np.cos(0.65), np.sin(0.65)])
ax.plot([v0[0], -v0[0]], [v0[1], -v0[1]], color="#6D597A", linewidth=3)
ax.scatter([v0[0], -v0[0]], [v0[1], -v0[1]], color="#6D597A")
ax.text(0, -1.25, "x and -x denote one axis", ha="center", fontsize=9)
ax.set_aspect("equal"); ax.set_title("RP: antipodal axes")
ax.set_xticks([]); ax.set_yticks([])

# SO(3) frame
ax = fig.add_subplot(2, 3, 4, projection="3d")
angle = 0.99
Rz = np.array([[np.cos(angle), -np.sin(angle), 0.0], [np.sin(angle), np.cos(angle), 0.0], [0.0, 0.0, 1.0]])
colors = ["#E76F51", "#2A9D8F", "#345995"]
for j in range(3):
    ax.quiver(0, 0, 0, Rz[0, j], Rz[1, j], Rz[2, j], color=colors[j], linewidth=2)
ax.set_xlim(-1,1); ax.set_ylim(-1,1); ax.set_zlim(0,1)
ax.set_title("SO(3): oriented frame")
ax.set_axis_off(); ax.set_box_aspect((1,1,0.7))

# Stiefel two-frame
ax = fig.add_subplot(2, 3, 5, projection="3d")
q1 = np.array([0.75, 0.35, 0.56]); q1 = q1/np.linalg.norm(q1)
q2 = np.array([-0.45, 0.89, 0.0]); q2 = q2 - q1*np.dot(q1, q2); q2 = q2/np.linalg.norm(q2)
ax.quiver(0,0,0,*q1,color="#E76F51",linewidth=2)
ax.quiver(0,0,0,*q2,color="#2A9D8F",linewidth=2)
ax.text(0,0,-0.25, f"dot={np.dot(q1,q2):.2g}", ha="center")
ax.set_xlim(-1,1); ax.set_ylim(-1,1); ax.set_zlim(-0.2,1)
ax.set_title("V_2(R^3): orthonormal columns")
ax.set_axis_off(); ax.set_box_aspect((1,1,0.8))

# Grassmann projection
ax = fig.add_subplot(2, 3, 6)
xx = np.linspace(-1.1, 1.1, 2)
ax.plot(xx, 0.45*xx, color="#577590", linewidth=3, label="subspace")
pt = np.array([0.35, 0.9])
base = np.array([1.0, 0.45]); base = base / np.linalg.norm(base)
proj = np.dot(pt, base) * base
ax.scatter(*pt, color="#E76F51")
ax.scatter(*proj, color="#2A9D8F")
ax.plot([pt[0], proj[0]], [pt[1], proj[1]], linestyle="--", color="#555555")
ax.set_aspect("equal"); ax.set_title("G_1(R^2): projection represents a line")
ax.set_xticks([]); ax.set_yticks([])

fig.suptitle("Notation sample-space gallery: constraints before coordinates", y=1.02, fontsize=14)
fig.tight_layout()
gallery_path = save_matplotlib(fig, TOPIC, "sample-spaces", "sample-space-constraint-gallery.png")
plt.close(fig)
display_artifact(gallery_path, width=940)

constraint_checks = {
    "stiefel_dot_abs": float(abs(np.dot(q1, q2))),
    "so3_orthogonality_error": float(np.linalg.norm(Rz.T @ Rz - np.eye(3))),
    "so3_determinant": float(np.linalg.det(Rz)),
    "grassmann_projection_idempotent_error": float(np.linalg.norm(np.outer(base, base) @ np.outer(base, base) - np.outer(base, base))),
}
constraint_checks



## Distribution Families As A Routing Table

The distribution names in the notation appendix are practical choices. A von Mises model belongs on the circle; a von Mises-Fisher model belongs on a sphere; Watson and Bingham models are axial; shape-analysis models must respect translation, scale, rotation, and sometimes complex phase. The table below is generated from structured records, so the notebook can check that every family names a support and a diagnostic.


In [ ]:

table_path = artifact_path(TOPIC, "tables", "distribution-family-routing.csv")
with table_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["family", "support", "parameters", "diagnostic", "chapters"])
    writer.writeheader()
    writer.writerows(distribution_families)

glossary_path = artifact_path(TOPIC, "tables", "notation-atlas.json")
glossary_path.write_text(json.dumps({
    "source_span": source_span,
    "sample_spaces": sample_spaces,
    "notation_entries": notation_entries,
    "distribution_families": distribution_families,
}, indent=2), encoding="utf-8")

display_artifact(table_path)
display_artifact(glossary_path)

routing_checks = {
    "distribution_family_count": len(distribution_families),
    "all_families_have_support": bool(all(row["support"] for row in distribution_families)),
    "all_families_have_diagnostic": bool(all(row["diagnostic"] for row in distribution_families)),
    "all_notation_entries_have_invariant": bool(all(row["invariant"] for row in notation_entries)),
    "atlas_json_bytes": glossary_path.stat().st_size,
}
routing_checks



## Interactive Notation Graph

The static dependency map gives the course-level structure. The interactive version below is smaller and hoverable. It keeps only the main notation entries, sample-space symbols, and distribution families, then exposes meaning and invariance in hover text. This is useful when the appendix is used as a reference while reading another chapter.

A good use pattern is to search for a symbol, then ask whether it denotes a population quantity, sample statistic, model family, or sample space. The hover text gives the interpretation and the invariant. The linked JSON file above is the durable machine-readable version.


In [ ]:

H = nx.Graph()
for item in sample_spaces:
    H.add_node(item["symbol"], group="sample space", hover=f"{item['name']}<br>{item['constraint']}<br>Invariant: {item['invariant']}")
for item in notation_entries:
    H.add_node(item["symbol"], group=item["category"], hover=f"{item['meaning']}<br>Object: {item['object']}<br>Invariant: {item['invariant']}<br>Use: {item['chapters']}")
for family in distribution_families:
    H.add_node(family["family"], group="distribution", hover=f"Support: {family['support']}<br>Parameters: {family['parameters']}<br>Diagnostic: {family['diagnostic']}")
    if family["support"] in H:
        H.add_edge(family["family"], family["support"])
for item in notation_entries:
    if item["category"] == "special-function":
        H.add_edge(item["symbol"], "von Mises-Fisher")
        H.add_edge(item["symbol"], "von Mises")
    elif item["category"] == "shape":
        H.add_edge(item["symbol"], "Sigma_m^k")
    elif item["object"] in H:
        H.add_edge(item["symbol"], item["object"])
    else:
        H.add_edge(item["symbol"], "S^1" if "S^1" in item["object"] else "S^(p-1)")

pos2 = nx.spring_layout(H, seed=444, k=0.9, iterations=120)
edge_x = []
edge_y = []
for a, b in H.edges():
    edge_x += [pos2[a][0], pos2[b][0], None]
    edge_y += [pos2[a][1], pos2[b][1], None]
edge_trace = go.Scatter(x=edge_x, y=edge_y, mode="lines", line=dict(width=1, color="#B0BEC5"), hoverinfo="none")

groups = sorted({H.nodes[n]["group"] for n in H.nodes})
palette = {group: color for group, color in zip(groups, ["#2A9D8F", "#E76F51", "#6D597A", "#577590", "#F4A261", "#8AB17D"])}
node_traces = []
for group in groups:
    nodes = [n for n in H.nodes if H.nodes[n]["group"] == group]
    node_traces.append(go.Scatter(
        x=[pos2[n][0] for n in nodes],
        y=[pos2[n][1] for n in nodes],
        mode="markers+text",
        text=nodes,
        textposition="top center",
        marker=dict(size=13, color=palette[group], line=dict(width=1, color="white")),
        hovertext=[H.nodes[n]["hover"] for n in nodes],
        hoverinfo="text",
        name=group,
    ))

interactive_graph = go.Figure(data=[edge_trace, *node_traces])
interactive_graph.update_layout(
    title="Appendix 04 interactive notation atlas",
    showlegend=True,
    margin=dict(l=10, r=10, t=50, b=10),
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    height=660,
)
interactive_path = save_plotly_html(interactive_graph, TOPIC, "interactive", "notation-atlas-graph.html", include_plotlyjs=True)
display_artifact(interactive_path, width="100%", height=680)

interactive_checks = {
    "interactive_node_count": H.number_of_nodes(),
    "interactive_edge_count": H.number_of_edges(),
    "all_interactive_nodes_have_hover": bool(all(H.nodes[n].get("hover") for n in H.nodes)),
}
interactive_checks



## Reader Checklist

Use this appendix whenever a symbol feels overloaded. Do not begin by asking what a letter stands for in isolation. Ask what type of object it is.

- If the symbol is a **sample space**, identify the constraint first: unit norm, orthogonality, projection idempotence, shape normalization, or antipodal identification.
- If the symbol is a **population quantity**, identify the transformation rule: scalar invariance for concentration, equivariance for mean direction, or moment behavior for resultant length.
- If the symbol is a **sample statistic**, identify the raw data operation and the range of valid values.
- If the symbol is a **distribution family**, identify the support before naming the parameters.
- If the symbol is a **special function**, identify the calculator job: normalizing a density, inverting concentration, or computing an axial moment.

This checklist is the reason the notebook saves both pictures and structured data. The pictures support visual memory; the JSON and CSV support auditability and reuse by later notebooks.


In [ ]:

numeric_checks = {
    **graph_diagnostics,
    **constraint_checks,
    **routing_checks,
    **interactive_checks,
    "source_span": source_span,
    "all_boolean_checks_pass": bool(
        all(value for value in graph_diagnostics.values() if isinstance(value, bool))
        and all(value for value in routing_checks.values() if isinstance(value, bool))
        and all(value for value in interactive_checks.values() if isinstance(value, bool))
    ),
}
assert numeric_checks["all_boolean_checks_pass"], numeric_checks
assert constraint_checks["stiefel_dot_abs"] < 1e-12
assert constraint_checks["so3_orthogonality_error"] < 1e-12
assert abs(constraint_checks["so3_determinant"] - 1.0) < 1e-12
assert constraint_checks["grassmann_projection_idempotent_error"] < 1e-12
checks_path = save_json(numeric_checks, TOPIC, "checks", "notation-atlas-checks.json")
display_artifact(checks_path)
numeric_checks


## Standalone Reading Notes

Treat this notation appendix as a type checker for the course. Before using a symbol in a computation, ask which row of the atlas it belongs to. A population mean direction is not a sample resultant. A concentration parameter is not a probability. A projective axis is not an oriented vector. A Stiefel frame is not a free rectangular matrix. A shape coordinate is not a raw landmark array until translation, scale, and rotation have been handled.

The graph and gallery are included because notation errors often begin visually. If two antipodal points are actually one projective observation, a mean-direction arrow can be misleading. If a frame is supposed to be orthonormal, a regression step that breaks `Q.T @ Q = I` has left the sample space. If a shape statistic changes after translating every landmark, it is no longer a shape statistic. These are not cosmetic distinctions; they decide which formulas from the rest of the book are valid.

The structured JSON is the reusable part. Later notebooks can point to the same symbol families, supports, diagnostics, and invariance rules instead of redefining them casually. When adding notation, add the support and invariant at the same time as the symbol. A glossary entry without those fields is too weak for a computational course.



## Takeaways

Appendix 04 is a map of geometric commitments.

- Sample-space symbols tell the reader which transformations are allowed and which coordinate operations are unsafe.
- Population and sample symbols become meaningful only after their support and invariance rules are named.
- Distribution-family names are routing decisions: circle, sphere, projective axis, frame manifold, subspace, or shape space.
- Special-function symbols are calculator labels. They normalize densities, invert concentration, and connect moments to parameters.
- Shape-analysis notation carries its own nuisance transformations: translation, scale, rotation, and sometimes complex phase.

A standalone directional-statistics course needs this notation atlas because later computations can otherwise look like ordinary array algebra. The atlas keeps the geometry attached to each symbol.


In [ ]:

final_sanity = {
    "source_span": source_span,
    "artifacts": assert_artifacts(
        [map_path, gallery_path, table_path, glossary_path, interactive_path, checks_path],
        min_bytes=100,
    ),
    "core_checks": {
        "all_notation_entries_have_course_use": graph_diagnostics["all_notation_entries_have_course_use"],
        "all_sample_spaces_have_constraints": graph_diagnostics["all_sample_spaces_have_constraints"],
        "all_families_have_support": routing_checks["all_families_have_support"],
        "all_families_have_diagnostic": routing_checks["all_families_have_diagnostic"],
        "all_interactive_nodes_have_hover": interactive_checks["all_interactive_nodes_have_hover"],
    },
    "pdf_used_for": "source orientation only; no copied glossary table, screenshots, or page crops",
    "standalone_contract": "notation dependency map, sample-space gallery, structured atlas, interactive graph, and invariant checks",
}
assert all(final_sanity["core_checks"].values()), final_sanity
final_path = save_json(final_sanity, TOPIC, "checks", "final-sanity.json")
assert final_path.exists() and final_path.stat().st_size > 100
final_sanity
